# 4. Testing the model

The two models are now tried on the test set. This shows how the models would have done on real applicants who came after the loans they were built on.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from optbinning import BinningProcess

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

test_36 = pd.read_parquet(PROCESSED / "test_36.parquet")
test_60 = pd.read_parquet(PROCESSED / "test_60.parquet")

binning_36 = BinningProcess.load(str(PROCESSED / "binning_36.pkl"))
binning_60 = BinningProcess.load(str(PROCESSED / "binning_60.pkl"))

weights_36 = pd.read_json(PROCESSED / "weights_36.json", typ="series")
weights_60 = pd.read_json(PROCESSED / "weights_60.json", typ="series")

features = [c for c in test_36.columns if c != "target"]

### 4.1 A PD for every test loan

Each test loan gets a PD in three steps, exactly as in the methodology. Its values are replaced by the WoE of their bins, using the bins found on the train set. The WoE values are multiplied by the weights and added up into a score. The score is turned into a PD with the logistic function.

In [2]:
def predict_pd(loans, binning, weights):
    woe = binning.transform(loans[features], metric="woe")
    chosen = [c for c in weights.index if c != "const"]
    score = weights["const"] + woe[chosen] @ weights[chosen]
    return 1 / (1 + np.exp(-score))


pd_36 = predict_pd(test_36, binning_36, weights_36)
pd_60 = predict_pd(test_60, binning_60, weights_60)

for name, p, test in [("36-month", pd_36, test_36), ("60-month", pd_60, test_60)]:
    print(f"{name}: average PD {p.mean():.1%}, default rate {test['target'].mean():.1%}")

36-month: average PD 13.4%, default rate 14.9%
60-month: average PD 26.5%, default rate 24.8%


The 36-month model predicts 13.4% on average, against 14.9% of loans that actually defaulted. The 60-month model predicts 26.5%, against 24.8%.

### 4.2 Does the model rank borrowers correctly?

Borrowers who end up defaulting should get higher PDs than borrowers who repay. The Gini says how well the model does this. It is 0 if the model is no better than guessing and 1 if it is perfect.

In [3]:
from sklearn.metrics import roc_auc_score

for name, p, test in [("36-month", pd_36, test_36), ("60-month", pd_60, test_60)]:
    auc = roc_auc_score(test["target"], p)
    print(f"{name}: AUC {auc:.3f}, Gini {2 * auc - 1:.3f}")

36-month: AUC 0.654, Gini 0.307
60-month: AUC 0.607, Gini 0.214


For the 36-month loans the Gini on the test set is 0.307, the same as on the train set (0.306). The model ranks new borrowers as well as the ones it was built on.

For the 60-month loans the Gini falls from 0.253 to 0.214. The model still ranks borrowers, but less well than on its train set. It was built on about a tenth as many loans, so it is less reliable.

### 4.3 Are the PDs accurate?

The test loans are sorted by PD and cut into 10 groups of the same size, from safest to riskiest. For each group, the table shows the average PD the model gave and the share of loans that actually defaulted. If the PDs are accurate, the two are close in every group.

In [4]:
def pd_groups(p, y):
    groups = pd.qcut(p, q=10, duplicates="drop")
    table = pd.DataFrame({"pd": p, "target": y}).groupby(groups, observed=True).agg(
        loans=("target", "count"),
        defaults=("target", "sum"),
        average_pd=("pd", "mean"),
        default_rate=("target", "mean"),
    )
    return table.round(3)

In [5]:
pd_groups(pd_36, test_36["target"])

,loans,defaults,average_pd,default_rate
"(0.013999999999999999, 0.0584]",28303,1193,0.044,0.042
"(0.0584, 0.078]",28303,2155,0.069,0.076
"(0.078, 0.0941]",28302,2735,0.086,0.097
"(0.0941, 0.109]",28303,3325,0.102,0.117
"(0.109, 0.125]",28302,3808,0.117,0.135
"(0.125, 0.141]",28303,4220,0.133,0.149
"(0.141, 0.161]",28302,4916,0.151,0.174
"(0.161, 0.185]",28303,5610,0.172,0.198
"(0.185, 0.222]",28302,6328,0.202,0.224
"(0.222, 0.476]",28303,7842,0.265,0.277


In [6]:
pd_groups(pd_60, test_60["target"])

,loans,defaults,average_pd,default_rate
"(0.0661, 0.163]",2214,290,0.135,0.131
"(0.163, 0.194]",2214,362,0.179,0.164
"(0.194, 0.218]",2219,434,0.206,0.196
"(0.218, 0.24]",2207,512,0.229,0.232
"(0.24, 0.261]",2214,538,0.250,0.243
"(0.261, 0.282]",2215,545,0.271,0.246
"(0.282, 0.305]",2215,606,0.293,0.274
"(0.305, 0.334]",2236,662,0.320,0.296
"(0.334, 0.372]",2184,709,0.352,0.325
"(0.372, 0.542]",2214,827,0.411,0.374


A model is calibrated when its PDs match what happens. Loans given a PD of 10% should default about 10% of the time.

For the 36-month loans, the default rate rises in every group, from 4.2% for the safest tenth to 27.7% for the riskiest, so the order is right. But the PDs are a little low in almost every group, by 1 to 2.6 points. The model says 13.4% overall, and 14.9% defaulted.

For the 60-month loans, the default rate rises from 13.1% to 37.4%. The PDs are accurate for the safer groups and 2 to 4 points too high for the riskier ones. The model says 26.5% overall, and 24.8% defaulted.

The cause is the same in both. Default rates changed between the train and test periods, and a model can only predict the level of risk it learned from. The order of the borrowers is right, but the level has drifted. In practice the PDs would be adjusted from time to time to match recent default rates.

### 4.4 Deciding whether to lend

The model gives each applicant a PD. The lender turns it into a decision with a cut-off. An applicant with a PD below the cut-off is approved, and one above it is declined.

The cut-off is a priority choice. Three ways of choosing it are shown here, for each loan term.

- By profit. Take the cut-off that would have earned the most. The profit of a loan is what the borrower paid back minus what was lent.
- By risk. Take the highest cut-off that keeps the default rate of the approved loans under a limit. The limit is 10% for the 36-month loans and 20% for the 60-month loans, which default about twice as often.
- By volume. Take the cut-off that approves a set share of applicants, here 85%.

The cut-offs are chosen on the train set, and then tried on the test set.

In [7]:
def decision_table(p, y, profit, lent, cutoffs):
    rows = []
    for c in cutoffs:
        approved = (p < c).to_numpy()
        rows.append({
            "cutoff": round(c, 2),
            "approved": approved.mean(),
            "default_rate_approved": y[approved].mean(),
            "defaults_avoided": y[~approved].sum() / y.sum(),
            "profit_million": profit[approved].sum() / 1e6,
            "return_on_lent": profit[approved].sum() / lent[approved].sum(),
        })
    return pd.DataFrame(rows).round(3)


def choose_cutoffs(table, risk_limit, volume):
    return {
        "By profit": table.loc[table["profit_million"].idxmax(), "cutoff"],
        "By risk": table.loc[table["default_rate_approved"] <= risk_limit, "cutoff"].max(),
        "By volume": table.loc[(table["approved"] - volume).abs().idxmin(), "cutoff"],
    }


def lending_results(group, binning, weights, test, test_pd, cutoffs, risk_limit, volume=0.85):
    train = pd.read_parquet(PROCESSED / f"train_{group}.parquet")
    train_pd = predict_pd(train, binning, weights)
    pay_train = pd.read_parquet(PROCESSED / f"payments_train_{group}.parquet")
    pay_test = pd.read_parquet(PROCESSED / f"payments_test_{group}.parquet")

    train_table = decision_table(train_pd, train["target"],
                                 pay_train["total_pymnt"] - pay_train["funded_amnt"],
                                 pay_train["funded_amnt"], cutoffs)
    chosen = choose_cutoffs(train_table, risk_limit, volume)

    test_table = decision_table(test_pd, test["target"],
                                pay_test["total_pymnt"] - pay_test["funded_amnt"],
                                pay_test["funded_amnt"], list(chosen.values()))
    test_table.index = list(chosen.keys())
    return train_table, test_table

#### 36-month loans

The first table shows what each cut-off would have done on the train set, for every fourth cut-off tried. The second shows the three chosen cut-offs applied to the test set.

In [14]:
train_table_36, result_36 = lending_results("36", binning_36, weights_36, test_36, pd_36,
                                            cutoffs=np.arange(0.04, 0.42, 0.01), risk_limit=0.10)
train_table_36.iloc[::4]

,cutoff,approved,default_rate_approved,defaults_avoided,profit_million,return_on_lent
0,0.04,0.036,0.027,0.992,12.610,0.081
4,0.08,0.224,0.056,0.904,94.926,0.093
8,0.12,0.489,0.080,0.700,214.722,0.100
12,0.16,0.716,0.099,0.456,310.216,0.102
16,0.20,0.861,0.113,0.255,363.600,0.101
20,0.24,0.941,0.122,0.122,387.560,0.100
24,0.28,0.978,0.127,0.050,397.468,0.099
28,0.32,0.993,0.129,0.017,400.602,0.098
32,0.36,0.998,0.130,0.005,401.330,0.098
36,0.40,1.000,0.131,0.001,401.466,0.098


In [10]:
result_36

,cutoff,approved,default_rate_approved,defaults_avoided,profit_million,return_on_lent
By profit,0.38,0.999,0.149,0.003,237.440,0.066
By risk,0.16,0.697,0.113,0.473,190.642,0.072
By volume,0.20,0.847,0.129,0.269,218.925,0.069


#### 60-month loans

The same two tables for the 60-month loans, with a risk limit of 20%.

In [11]:
train_table_60, result_60 = lending_results("60", binning_60, weights_60, test_60, pd_60,
                                            cutoffs=np.arange(0.08, 0.58, 0.01), risk_limit=0.20)
train_table_60.iloc[::4]

,cutoff,approved,default_rate_approved,defaults_avoided,profit_million,return_on_lent
0,0.08,0.005,0.046,0.999,0.517,0.171
4,0.12,0.045,0.093,0.984,4.620,0.181
8,0.16,0.141,0.124,0.931,16.281,0.195
12,0.20,0.283,0.156,0.826,33.369,0.194
16,0.24,0.460,0.179,0.676,56.512,0.199
20,0.28,0.632,0.203,0.495,76.851,0.195
24,0.32,0.774,0.222,0.323,92.591,0.190
28,0.36,0.881,0.235,0.184,105.073,0.188
32,0.40,0.948,0.244,0.089,111.873,0.185
36,0.44,0.983,0.250,0.032,114.720,0.182


In [12]:
result_60

,cutoff,approved,default_rate_approved,defaults_avoided,profit_million,return_on_lent
By profit,0.52,1.000,0.248,0.000,87.640,0.199
By risk,0.27,0.540,0.196,0.572,50.880,0.213
By volume,0.35,0.847,0.228,0.222,76.883,0.206


#### Summary

Each way of choosing gives a different cut-off.

- By profit, almost everyone is approved. Riskier borrowers paid more interest, so even risky loans made money on average.
- By risk, the cut-off is the strictest. It declines about a third of the 36-month applicants and nearly half of the 60-month ones, and these hold about half of all the defaults.
- By volume, it sits in between. It approves 85% of applicants and declines about a quarter of the defaults.

So there is a trade-off. A stricter cut-off means fewer defaults and a slightly better return on each dollar lent, but less profit in total.

One caution. The 36-month risk cut-off aimed for a default rate of 10% and got 11.3% on the test set, because loans from 2015 defaulted more than earlier ones. A rule chosen on past loans has to be watched and adjusted.